In [0]:
%run ./addressParse

In [0]:
"""Pipeline configuration for Bronze storage location ingest."""
import re
from datetime import datetime, date, timezone
from typing import Any
from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType, IntegerType

CATALOG = "bis_dev"
BRONZE_SCHEMA = "bronze_roster"
SILVER_SCHEMA = "silver_roster"

VOLUME_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/manual_inputs"
CHECKPOINT = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/manual_inputs/_checkpoints/emp_storage_location"

# Strict glob for structural isolation, since manual_inputs is a shared drop zone
FILE_GLOB = "*[Ss]torage*[Ss]pace*.xlsx"
SHEET_NAME = "Spaces"

BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.emp_storage_location"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.emp_storage_location"

ROSTER_TABLE = None  # set to enable the two email-match QA checks
QA_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.roster_qa"
AUDIT_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.autoloader_audit_log"

# source header -> (target column, nullable)
COLUMN_SPEC: dict[str, tuple[str, bool]] = {
    "RepName":          ("Rep_Name",             False),
    "Rep Email":        ("Rep_Email",            False),
    "Rep Phone":        ("Rep_Phone",            False),
    "Territory ID":     ("Territory_ID",         False),
    "Emp ID":           ("Emp_ID",               False),
    "Facility Name":    ("Facility_Name",        False),
    "Facility Address": ("Facility_Raw_Address", False),
    "Space Number":     ("Space_Number",         False),
    "Size":             ("Storage_Size",         False),
    "Territory Name":   ("Territory_Name",       False),
    "Director":         ("Region_Emp_Name",      True),
    "Region Name":      ("Region_Name",          True),
}

# Column order here is the write contract. load_batch builds tuples from
# [f.name for f in BRONZE_STRUCT.fields], so the two can never drift.
BRONZE_STRUCT = StructType(
    [StructField(target, StringType(), True) for target, _ in COLUMN_SPEC.values()]
    + [StructField(c, StringType(), True) for c in PARSED_COLUMNS]
    + [
        StructField("File_Date", DateType(), True),
        StructField("File_Name", StringType(), True),
        StructField("Load_Timestamp", TimestampType(), True),
        StructField("_rescued_data", StringType(), True),
    ]
)


def extract_file_date(file_name: str | None, fallback_date: date) -> date:
    """Extract date from filename (strict US format MMDDYYYY, MM_DD_YYYY, MM-DD-YYYY, MM/DD/YYYY) falling back to UTC modificationTime."""
    if not file_name:
        return fallback_date

    # Delimited US date formats: MM-DD-YYYY, MM_DD_YYYY, MM/DD/YYYY, MM.DD.YYYY
    m_us = re.search(r"(0[1-9]|1[0-2])[-_./\s](0[1-9]|[12]\d|3[01])[-_./\s](19|20)\d{2}", file_name)
    if m_us:
        s = re.sub(r"[-_./\s]", "", m_us.group(0))
        return datetime.strptime(s, "%m%d%Y").date()

    # 8-digit consecutive US date format: MMDDYYYY
    for m in re.finditer(r"\d{8}", file_name):
        s = m.group(0)
        try:
            dt = datetime.strptime(s, "%m%d%Y").date()
            if 1900 <= dt.year <= 2100:
                return dt
        except ValueError:
            pass

    return fallback_date


print(f"✓ Schema configured: {len(BRONZE_STRUCT.fields)} total columns")
print(f"  - {len(COLUMN_SPEC)} source columns")
print(f"  - {len(PARSED_COLUMNS)} parsed columns") 
print(f"  - 4 metadata columns")
print(f"\nTarget: {BRONZE_TABLE}")

In [0]:
import json
from io import BytesIO
from typing import Any
from openpyxl import load_workbook
from openpyxl.workbook import Workbook
from openpyxl.worksheet.worksheet import Worksheet


def clean(value: Any) -> str | None:
    """Normalize cell value to trimmed string or None."""
    if value is None:
        return None
    return str(value).strip() or None


def select_sheet(workbook: Workbook, file_name: str) -> tuple[Worksheet, str | None]:
    """Select target worksheet, falling back to sole visible sheet if absent."""
    try:
        return workbook[SHEET_NAME], None
    except KeyError:
        visible = [ws.title for ws in workbook.worksheets if ws.sheet_state == "visible"]
        if len(visible) == 1:
            return workbook[visible[0]], (
                f"sheet '{SHEET_NAME}' missing, used sole visible sheet '{visible[0]}'"
            )
        raise ValueError(
            f"Sheet '{SHEET_NAME}' not found in {file_name} and fallback is ambiguous. "
            f"Visible sheets: {visible}"
        )


def classify_header(header: tuple[Any, ...]) -> tuple[dict[str, int], dict[int, str], list[str]]:
    """Map header row to target columns and collect unmapped header names."""
    def key(h: Any) -> str:
        return str(h).strip().lower() if h is not None else ""

    seen = {key(h): i for i, h in enumerate(header) if key(h)}
    index_by_target, missing, matched = {}, [], set()

    for source, (target, _) in COLUMN_SPEC.items():
        position = seen.get(key(source))
        if position is None:
            missing.append(target)
        else:
            index_by_target[target] = position
            matched.add(position)
    
    rescued_names = {
        i: (clean(header[i]) or f"_c{i}")
        for i in range(len(header))
        if i not in matched
    }
    return index_by_target, rescued_names, missing


def read_workbook(
    content: bytes, file_name: str, file_date: date, load_ts: datetime
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    """Decode Excel bytes into parsed records conforming to BRONZE_STRUCT."""
    with load_workbook(BytesIO(content), read_only=True, data_only=True) as workbook:
        sheet, sheet_note = select_sheet(workbook, file_name)
        rows = list(sheet.iter_rows(values_only=True))

    if not rows:
        raise ValueError(f"Sheet '{SHEET_NAME}' is empty in {file_name}")

    header = rows[0]
    index_by_target, rescued_names, missing = classify_header(header)

    if missing:
        raise ValueError(f"{file_name} missing column(s): {missing}")

    records = []

    for raw_row in rows[1:]:
        values = [clean(v) for v in raw_row]
        if not any(values):
            continue

        record = {
            target: (values[i] if i < len(values) else None)
            for target, i in index_by_target.items()
        }

        rescued = {}
        for i, value in enumerate(values):
            if value is None:
                continue
            if i in rescued_names:
                rescued[rescued_names[i]] = value
            elif i >= len(header):
                rescued[f"_c{i}"] = value
        
        record.update(parse_address(record.get("Facility_Raw_Address")))
        record["File_Date"] = file_date
        record["File_Name"] = file_name
        record["Load_Timestamp"] = load_ts
        record["_rescued_data"] = json.dumps(rescued) if rescued else None
        records.append(record)

    notes = {"sheet": sheet_note, "rescued_columns": list(rescued_names.values())}
    return records, notes

In [0]:
from datetime import datetime, timezone

# for autoloader logs
_AUDIT_SCHEMA = StructType([
    StructField("run_ts", TimestampType(), False),
    StructField("batch_id", IntegerType(), False),
    StructField("files_processed", IntegerType(), False),
    StructField("rows_written", IntegerType(), False),
    StructField("file_names", StringType(), True),
    StructField("status", StringType(), False),
    StructField("notes", StringType(), True),
])


def _log_audit(
    run_ts: datetime,
    batch_id: int,
    files_processed: int,
    rows_written: int,
    file_names: str | None,
    status: str,
    notes: str | None = None,
) -> None:
    """Append row to audit log table."""
    row = [(run_ts, batch_id, files_processed, rows_written, file_names, status, notes)]
    spark.createDataFrame(row, schema=_AUDIT_SCHEMA).write.mode("append").saveAsTable(AUDIT_TABLE)


def load_batch(batch_df: Any, batch_id: int) -> None:
    """Process micro-batch of files, parse addresses, and append to Bronze."""
    records, notes = [], []
    load_ts = datetime.now(timezone.utc)
    rows = batch_df.select("path", "content", "modificationTime").collect()
    file_count = len(rows)
    print(f"[{batch_id}] batch received: {file_count} file(s)")

    for row in rows:
        file_name = row["path"].rsplit("/", 1)[-1]
        raw_mod_dt = row["modificationTime"]
        raw_mod_utc_date = raw_mod_dt.astimezone(timezone.utc).date() if hasattr(raw_mod_dt, "tzinfo") and raw_mod_dt.tzinfo else raw_mod_dt.date()
        file_date = extract_file_date(file_name, raw_mod_utc_date)
        try:
            file_records, file_notes = read_workbook(row["content"], file_name, file_date, load_ts)
        except Exception as e:
            print(f"[{batch_id}] FAILED: {file_name} — {type(e).__name__}: {e}")
            try:
                _log_audit(load_ts, batch_id, len(notes), 0, file_name, "FAILED", f"{type(e).__name__}: {e}")
            except Exception as ae:
                print(f"[{batch_id}] WARN: audit log write failed: {ae}")
            raise
        records.extend(file_records)
        notes.append((file_name, len(file_records), file_notes))

    if not records:
        try:
            _log_audit(load_ts, batch_id, file_count, 0, None, "EMPTY", "Files contained no extractable data rows")
        except Exception as e:
            print(f"[{batch_id}] WARN: audit log write failed: {e}")
        return

    columns = [f.name for f in BRONZE_STRUCT.fields]
    tuples = [tuple(r.get(c) for c in columns) for r in records]

    (
        spark.createDataFrame(tuples, schema=BRONZE_STRUCT)
        .write.mode("append")
        .saveAsTable(BRONZE_TABLE)
    )

    file_names_str = ", ".join(fn for fn, _, _ in notes)
    try:
        _log_audit(load_ts, batch_id, file_count, len(tuples), file_names_str, "SUCCESS")
    except Exception as e:
        print(f"[{batch_id}] WARN: audit log write failed: {e}")

    for file_name, count, file_notes in notes:
        print(f"[{batch_id}] {file_name}: {count} rows | {file_notes}")

In [0]:
from datetime import datetime, timezone

stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")
    .option("pathGlobFilter", FILE_GLOB)
    .load(VOLUME_PATH)
    .writeStream.option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .foreachBatch(load_batch)
    .start()
)
stream.awaitTermination()

# for autoloader logs
total_input_rows = sum(
    (s.get("numInputRows") or 0)
    for p in stream.recentProgress
    for s in (p.get("sources") or [])
)
num_batches = len(stream.recentProgress)

if total_input_rows == 0:
    try:
        _log_audit(datetime.now(timezone.utc), -1, 0, 0, None, "NO_NEW_FILES", "Stream found no unprocessed files")
    except Exception as e:
        print(f"WARN: audit log write failed: {e}")
    print("\u2139\ufe0f  No new files found since last checkpoint — nothing to process.")
else:
    print(f"\u2705  Stream completed: {total_input_rows} file(s) across {num_batches} batch(es).")

print(f"\nQuery audit log: SELECT * FROM {AUDIT_TABLE} ORDER BY run_ts DESC LIMIT 10")

In [0]:
from datetime import datetime, timezone

QA_RESULT_SCHEMA = ("run_ts timestamp, layer string, file_date date, "
                    "check_name string, severity string, fail_count int, description string")


def run_qa(layer: str, checks: list[tuple[str, str, str, str | None]]) -> Any:
    """Run QA checks, persist results to QA_TABLE, and report failures."""
    run_ts = datetime.now(timezone.utc)
    file_date = spark.sql(f"SELECT MAX(File_Date) FROM {BRONZE_TABLE}").collect()[0][0]
    rows = []

    for name, severity, description, sql in checks:
        if sql is None:
            rows.append((run_ts, layer, file_date, name, "SKIPPED", 0, description))
            continue
        try:
            count = int(spark.sql(sql).collect()[0][0] or 0)
            rows.append((run_ts, layer, file_date, name, severity, count, description))
        except Exception as e:
            rows.append((run_ts, layer, file_date, name, "CHECK_FAILED", -1, f"{description} | {e}"))

    df = spark.createDataFrame(rows, QA_RESULT_SCHEMA)
    df.write.mode("append").saveAsTable(QA_TABLE)

    failures = [r for r in rows if r[5] != 0]
    skipped  = [r for r in rows if r[4] == "SKIPPED"]

    for r in failures:
        print(f"[{r[4]}] {r[3]} = {r[5]} — {r[6]}")
    for r in skipped:
        print(f"[SKIPPED] {r[3]} — not configured")

    if not failures:
        print(f"{layer}: {len(rows) - len(skipped)} passed, {len(skipped)} skipped.")

    return df

In [0]:
LATEST_FILE = f"(SELECT MAX(File_Date) FROM {BRONZE_TABLE})"

bronze_checks = [
    (
        "email_matches_roster",
        "ERROR",
        "Rep_Email does not match the roster, or Emp_ID is absent from it",
        None if ROSTER_TABLE is None else
        f"""SELECT COUNT(*) FROM {BRONZE_TABLE} b
            LEFT JOIN {ROSTER_TABLE} r ON b.Emp_ID = r.Emp_ID
            WHERE b.File_Date = {LATEST_FILE}
              AND (r.Emp_ID IS NULL OR lower(b.Rep_Email) <> lower(r.Rep_Email))""",
    ),
    (
        "one_location_per_emp_per_file",
        "WARNING",
        "Emp_ID appears more than once in the same file; Silver keeps the most recently loaded row",
        f"""SELECT COUNT(*) FROM (
              SELECT Emp_ID, File_Name FROM {BRONZE_TABLE}
              WHERE File_Date = {LATEST_FILE}
              GROUP BY Emp_ID, File_Name HAVING COUNT(*) > 1)""",
    ),
    (
        "rescued_data_present",
        "WARNING",
        "Source file contained a new, renamed or extra column",
        f"SELECT COUNT(*) FROM {BRONZE_TABLE} "
        f"WHERE File_Date = {LATEST_FILE} AND _rescued_data IS NOT NULL",
    ),
]

display(run_qa("BRONZE", bronze_checks))

In [0]:
%skip
%sql
REFRESH MATERIALIZED VIEW bis_dev.silver_roster.emp_storage_location;

In [0]:
%skip
silver_checks = [
    (
        "address_not_parsed",
        "WARNING",
        "Address_1 is blank or Parsing_Error is populated",
        f"SELECT COUNT(*) FROM {SILVER_TABLE} "
        f"WHERE Address_1 IS NULL OR Parsing_Error IS NOT NULL",
    ),
]

display(run_qa("SILVER", silver_checks))